**Column Transformers**

In [11]:
import pandas as pd
import numpy as np

In [12]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [13]:
df = pd.read_csv('covid_toy.csv')

In [14]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [21]:
df.dtypes

age            int64
gender        object
fever        float64
cough         object
city          object
has_covid     object
dtype: object

In [16]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2)

In [17]:
X_train

,age,gender,fever,cough,city
68,54,Female,104.0,Strong,Kolkata
84,69,Female,98.0,Strong,Mumbai
97,20,Female,101.0,Mild,Bangalore
58,23,Male,98.0,Strong,Mumbai
2,42,Male,101.0,Mild,Delhi
...,...,...,...,...,...
40,49,Female,102.0,Mild,Delhi
75,5,Male,102.0,Mild,Kolkata
15,70,Male,103.0,Strong,Kolkata
23,80,Female,98.0,Mild,Delhi


**1. Manually encoding and merging encoded columns to form a dataset**

In [22]:
#adding simple imputer to fever col
#SimpleImputer fills null values in a numerical column with the columns mean

si=SimpleImputer()

X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [23]:
#OrdinalEncoding --> cough

oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [ ]:
#OneHotEncoding --> gender, city

ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])
X_test_gender_city = ohe.fit_transform(X_test[['gender', 'city']])

X_train_gender_city.shape

(80, 4)

In [28]:
#Extracting Age

cols = ['gender', 'fever', 'cough', 'city']
X_train_age = X_train.drop(columns=cols).values
X_test_age = X_test.drop(columns=cols).values

In [29]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)
X_train_transformed.shape

(80, 7)

**2. Using column transformers**<br><br>

Saves lot of time by preventing manual encoding and merging

In [30]:
from sklearn.compose import ColumnTransformer

In [32]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
], remainder='passthrough')

In [33]:
transformer.fit_transform(X_train).shape

(80, 7)

In [34]:
transformer.fit_transform(X_test).shape

(20, 7)